<a href="https://colab.research.google.com/github/kartik-sharma-0786/ML-in-python-freecodecamp-/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

categorical var into binary with no multicollnearity


In [ ]:
cat_cols = ['sex', 'smoker', 'region']
dataset=pd.get_dummies(dataset, columns=cat_cols, drop_first=True)
dataset.head()

#data division and assigning train_labels

In [ ]:

train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)
train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
train_stats = train_dataset.describe()
train_stats = train_stats.transpose()
train_stats

In [ ]:
def norm(x):
  return (x - train_stats['mean']) / train_stats['std']

# Identify numerical and categorical columns from the dataset before popping 'expenses'
numerical_cols = ['age', 'bmi', 'children']
# Get all columns from train_dataset that are not in numerical_cols
categorical_cols = [col for col in train_dataset.columns if col not in numerical_cols]

# Normalize numerical columns
normed_train_numerical = norm(train_dataset[numerical_cols])
normed_test_numerical = norm(test_dataset[numerical_cols])

# Combine normalized numerical columns with original categorical columns
normed_train_data = pd.concat([normed_train_numerical, train_dataset[categorical_cols]], axis=1)
normed_test_data = pd.concat([normed_test_numerical, test_dataset[categorical_cols]], axis=1)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential([
    layers.Dense(256, activation='relu', input_shape=[len(train_dataset.keys())]),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,activation='relu'),
    layers.Dense(32,activation='relu'),
    layers.Dense(16,activation='relu'),
    layers.Dense(1)
])
optimizer=tf.keras.optimizers.RMSprop()
model.compile(loss='mse',
              optimizer=optimizer,
              metrics=['mae', 'mse'])
model.summary()

In [ ]:
from keras import callbacks
epochs = 1000
history = model.fit(
  train_dataset, train_labels,
  epochs=epochs, validation_split = 0.2,verbose=1,callbacks=[tf.keras.callbacks.ModelCheckpoint("./checkpoint.keras", save_best_only=True, monitor='val_loss')])

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
